<a href="https://colab.research.google.com/github/kaltad16/Tri-line/blob/main/TRYALINE_CNN_GNN_Finetuning_v1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TRYALINE CNN + GNN fine-tuning on Colab GPU

This notebook fine-tunes both existing models independently using the search-labelled JSONL dataset. It exports new checkpoints and never overwrites the originals. This is supervised policy/value fine-tuning, **not full AlphaZero/MCTS training**.

Upload these three files from your project:
- `data/research/tryaline_train_v1.jsonl`
- `public/models/resnet_round12_adversarial_selfplay.json`
- `public/models/round22_adversarial_selfplay.json`

In Colab choose Runtime → Change runtime type → GPU.

In [3]:
from google.colab import files
import os, json, math, random, copy, time
import numpy as np
uploaded = files.upload()
dataset_path = next((k for k in uploaded if k.endswith(".jsonl")), None)
cnn_path = next((k for k in uploaded if "resnet" in k.lower() and k.endswith(".json")), None)
gnn_path = next((k for k in uploaded if "round22" in k.lower() and k.endswith(".json")), None)
if not dataset_path or not cnn_path or not gnn_path:
    raise ValueError("Upload the dataset JSONL and both original checkpoint JSON files.")
import torch
import torch.nn as nn
import torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "PyTorch:", torch.__version__)

Saving resnet_round12_adversarial_selfplay.json to resnet_round12_adversarial_selfplay.json
Saving round22_adversarial_selfplay.json to round22_adversarial_selfplay.json
Saving tryaline_train_v1.jsonl to tryaline_train_v1.jsonl
Device: cpu PyTorch: 2.11.0+cpu


## 1. Validate data and split by game

In [4]:
rows=[]
with open(dataset_path, encoding="utf-8") as f:
    rows=[json.loads(line) for line in f if line.strip()]
for i,r in enumerate(rows):
    n=int(r["boardSize"])**2
    assert len(r["features"])==n and all(len(x)==10 for x in r["features"]), f"features row {i}"
    assert len(r["adjacency"])==n, f"adjacency row {i}"
    assert r["candidates"] and 0<=int(r["actionIndex"])<len(r["candidates"]), f"policy row {i}"
    assert int(r["outcome"]) in (0,1,2), f"outcome row {i}"
games=sorted({int(r["gameId"]) for r in rows})
random.Random(20261015).shuffle(games)
val_ids=set(games[:max(1,round(.2*len(games)))])
train_rows=[r for r in rows if int(r["gameId"]) not in val_ids]
val_rows=[r for r in rows if int(r["gameId"]) in val_ids]
print(f"{len(rows)} positions, {len(games)} games; train {len(train_rows)}, validation {len(val_rows)}")
print("outcomes:", {k:sum(int(r["outcome"])==k for r in rows) for k in (0,1,2)})
cnn_json=json.load(open(cnn_path, encoding="utf-8"))
gnn_json=json.load(open(gnn_path, encoding="utf-8"))
assert "state" in cnn_json and "state" in gnn_json, "Checkpoint missing state object"
print("CNN keys:", list(cnn_json["state"])[:10])
print("GNN keys:", list(gnn_json["state"])[:10])

4296 positions, 74 games; train 3455, validation 841
outcomes: {0: 1006, 1: 2257, 2: 1033}
CNN keys: ['stem.weight', 'stem.bias', 'trunk.0.conv1.weight', 'trunk.0.conv1.bias', 'trunk.0.conv2.weight', 'trunk.0.conv2.bias', 'trunk.1.conv1.weight', 'trunk.1.conv1.bias', 'trunk.1.conv2.weight', 'trunk.1.conv2.bias']
GNN keys: ['encoder.weight', 'encoder.bias', 'layers.0.self_linear.weight', 'layers.0.self_linear.bias', 'layers.0.neighbor_linear.weight', 'layers.0.neighbor_linear.bias', 'layers.0.norm.weight', 'layers.0.norm.bias', 'layers.1.self_linear.weight', 'layers.1.self_linear.bias']


## 2. Architecture definitions
These definitions match the checkpoint tensor shapes: CNN `10→24`, three residual blocks, policy `97→64→1`, value `24→64→3`; GNN width 64 with three residual message-passing layers, policy `257→64→1`, value `64→64→1`.

In [5]:
class ResidualBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(24,24,3,padding=1)
        self.conv2=nn.Conv2d(24,24,3,padding=1)
    def forward(self,x):
        return F.relu(x+self.conv2(F.relu(self.conv1(x))))

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem=nn.Conv2d(10,24,3,padding=1)
        self.trunk=nn.ModuleList([ResidualBlock() for _ in range(3)])
        self.policy=nn.Sequential(nn.Linear(97,64),nn.ReLU(),nn.Linear(64,1))
        self.value=nn.Sequential(nn.Linear(24,64),nn.ReLU(),nn.Linear(64,3))
    def forward_sample(self,s):
        feat=torch.tensor(s["features"],dtype=torch.float32,device=device)
        n=feat.shape[0]; size=int(round(math.sqrt(n)))
        x=F.relu(self.stem(feat.reshape(1,size,size,10).permute(0,3,1,2)))
        for b in self.trunk: x=b(x)
        x=x[0].permute(1,2,0).reshape(n,24); pool=x.mean(0)
        acts=s["candidates"]
        fr=torch.tensor([int(a["from"])-1 for a in acts],device=device)
        to=torch.tensor([int(a["to"])-1 for a in acts],device=device)
        tp=torch.tensor([[float(a["teleport"])] for a in acts],device=device)
        inp=torch.cat([x[fr],x[to],x[fr]*x[to],pool.expand(len(acts),-1),tp],1)
        return self.policy(inp).squeeze(-1), self.value(pool)

class GNNLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.self_linear=nn.Linear(64,64)
        self.neighbor_linear=nn.Linear(64,64)
        self.norm=nn.LayerNorm(64,eps=1e-5)
    def forward(self,x,adj):
        ns=[]
        for i,neighbors in enumerate(adj):
            ns.append(x[torch.tensor(neighbors,dtype=torch.long,device=device)].mean(0) if neighbors else torch.zeros_like(x[i]))
        y=self.self_linear(x)+self.neighbor_linear(torch.stack(ns))
        return x+F.relu(self.norm(y))

class GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Linear(10,64)
        self.layers=nn.ModuleList([GNNLayer() for _ in range(3)])
        self.policy=nn.Sequential(nn.Linear(257,64),nn.ReLU(),nn.Linear(64,1))
        self.value=nn.Sequential(nn.Linear(64,64),nn.ReLU(),nn.Linear(64,1))
    def forward_sample(self,s):
        x=F.relu(self.encoder(torch.tensor(s["features"],dtype=torch.float32,device=device)))
        for layer in self.layers: x=layer(x,s["adjacency"])
        pool=x.mean(0); acts=s["candidates"]
        fr=torch.tensor([int(a["from"])-1 for a in acts],device=device)
        to=torch.tensor([int(a["to"])-1 for a in acts],device=device)
        tp=torch.tensor([[float(a["teleport"])] for a in acts],device=device)
        inp=torch.cat([x[fr],x[to],x[fr]*x[to],pool.expand(len(acts),-1),tp],1)
        return self.policy(inp).squeeze(-1), torch.tanh(self.value(pool).squeeze(-1))

In [6]:


print("Device:", device)
print("CNN loaded:", isinstance(cnn, CNN))
print("GNN loaded:", isinstance(gnn, GNN))
print("Dataset rows:", len(rows))
print("Validation rows:", len(val_rows))

sample = val_rows[0]
gnn.eval()

with torch.no_grad():
    logits, value = gnn.forward_sample(sample)

print("Board size:", sample["boardSize"])
print("Candidate count:", len(sample["candidates"]))
print("Reference logits:", logits.cpu().tolist()[:10])
print("Reference value:", float(value.cpu()))

Device: cpu


NameError: name 'cnn' is not defined

## 3. Strictly load existing checkpoint weights
If a tensor key or shape differs, this cell stops. It will not silently train from random initialization.

In [ ]:
def put(module, state, key):
    if key not in state: raise KeyError("Missing tensor: "+key)
    src=torch.tensor(state[key],dtype=torch.float32)
    if tuple(src.shape)!=tuple(module.shape): raise ValueError(f"{key}: {tuple(src.shape)} != {tuple(module.shape)}")
    with torch.no_grad(): module.copy_(src)

def load_cnn(m, ck):
    s=ck["state"]
    put(m.stem.weight,s,"stem.weight"); put(m.stem.bias,s,"stem.bias")
    for i,b in enumerate(m.trunk):
        for name in ("conv1","conv2"):
            l=getattr(b,name); put(l.weight,s,f"trunk.{i}.{name}.weight"); put(l.bias,s,f"trunk.{i}.{name}.bias")
    for name in ("policy","value"):
        h=getattr(m,name)
        for ix in (0,2):
            put(h[ix].weight,s,f"{name}.{ix}.weight"); put(h[ix].bias,s,f"{name}.{ix}.bias")

def load_gnn(m, ck):
    s=ck["state"]
    put(m.encoder.weight,s,"encoder.weight"); put(m.encoder.bias,s,"encoder.bias")
    for i,l in enumerate(m.layers):
        for name in ("self_linear","neighbor_linear"):
            q=getattr(l,name); put(q.weight,s,f"layers.{i}.{name}.weight"); put(q.bias,s,f"layers.{i}.{name}.bias")
        put(l.norm.weight,s,f"layers.{i}.norm.weight"); put(l.norm.bias,s,f"layers.{i}.norm.bias")
    for name in ("policy","value"):
        h=getattr(m,name)
        for ix in (0,2):
            put(h[ix].weight,s,f"{name}.{ix}.weight"); put(h[ix].bias,s,f"{name}.{ix}.bias")

cnn=CNN().to(device); gnn=GNN().to(device)
load_cnn(cnn,cnn_json); load_gnn(gnn,gnn_json)
print("Both existing checkpoints loaded successfully.")

Both existing checkpoints loaded successfully.


## 4. Fine-tuning
Three epochs is a smoke training run. Increase later only after inference compatibility and gameplay evaluation. Each game is kept entirely in one split.

In [ ]:
def epoch(model,data,kind,opt=None):
    train=opt is not None; model.train(train)
    random.shuffle(data) if train else None
    sums=np.zeros(3,dtype=float); count=0
    with (torch.enable_grad() if train else torch.no_grad()):
        for s in data:
            logits,val=model.forward_sample(s)
            pi=torch.tensor([int(s["actionIndex"])],dtype=torch.long,device=device)
            pl=F.cross_entropy(logits[None,:],pi)
            outcome=int(s["outcome"])
            if kind=="cnn":
                vl=F.cross_entropy(val[None,:],torch.tensor([outcome],dtype=torch.long,device=device))
            else:
                vl=F.smooth_l1_loss(val,torch.tensor(float(outcome-1),device=device))
            loss=pl+.5*vl
            if train:
                opt.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            sums += [float(pl.detach()),float(vl.detach()),float((logits.argmax().item()==int(s["actionIndex"])))]
            count+=1
    return {"policy_loss":sums[0]/max(1,count),"value_loss":sums[1]/max(1,count),"policy_top1":sums[2]/max(1,count),"n":count}

def fit(model,kind,epochs=3,lr=1e-4):
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-5)
    best=float("inf"); best_state=copy.deepcopy(model.state_dict()); history=[]
    for e in range(1,epochs+1):
        t=time.time(); tr=epoch(model,train_rows,kind,opt); va=epoch(model,val_rows,kind)
        score=va["policy_loss"]+.5*va["value_loss"]
        if score<best: best=score; best_state=copy.deepcopy(model.state_dict())
        result={"epoch":e,"train":tr,"validation":va,"validation_combined_loss":score,"seconds":round(time.time()-t,1)}
        history.append(result); print(json.dumps(result))
    model.load_state_dict(best_state)
    return history

cnn_history=fit(cnn,"cnn",epochs=3,lr=1e-4)

{"epoch": 1, "train": {"policy_loss": 2.317724453933339, "value_loss": 0.8682544497980653, "policy_top1": 0.3276410998552822, "n": 3455}, "validation": {"policy_loss": 2.2933375978291353, "value_loss": 0.48765967831559737, "policy_top1": 0.36028537455410226, "n": 841}, "validation_combined_loss": 2.537167436986934, "seconds": 22.4}
{"epoch": 2, "train": {"policy_loss": 2.0972903427674243, "value_loss": 0.6228362448821836, "policy_top1": 0.39623733719247467, "n": 3455}, "validation": {"policy_loss": 2.4173776340922153, "value_loss": 0.33939256050717337, "policy_top1": 0.37217598097502974, "n": 841}, "validation_combined_loss": 2.587073914345802, "seconds": 21.1}
{"epoch": 3, "train": {"policy_loss": 2.008851116122462, "value_loss": 0.5289480932639908, "policy_top1": 0.45354558610709117, "n": 3455}, "validation": {"policy_loss": 2.599190540660132, "value_loss": 0.3277644860289105, "policy_top1": 0.37098692033293695, "n": 841}, "validation_combined_loss": 2.763072783674587, "seconds": 21.

In [ ]:
gnn_history=fit(gnn,"gnn",epochs=3,lr=1e-4)

{"epoch": 1, "train": {"policy_loss": 2.566747053672712, "value_loss": 0.25541087004224644, "policy_top1": 0.20578871201157742, "n": 3455}, "validation": {"policy_loss": 2.474487319624098, "value_loss": 0.16354310817934647, "policy_top1": 0.262782401902497, "n": 841}, "validation_combined_loss": 2.5562588737137713, "seconds": 57.3}
{"epoch": 2, "train": {"policy_loss": 2.468509762205841, "value_loss": 0.2552390224807924, "policy_top1": 0.25470332850940663, "n": 3455}, "validation": {"policy_loss": 2.464372242522934, "value_loss": 0.16337385938370547, "policy_top1": 0.2615933412604043, "n": 841}, "validation_combined_loss": 2.546059172214787, "seconds": 55.7}
{"epoch": 3, "train": {"policy_loss": 2.4400273776423136, "value_loss": 0.2549708462867485, "policy_top1": 0.25991316931982633, "n": 3455}, "validation": {"policy_loss": 2.509285372246708, "value_loss": 0.16340872807143367, "policy_top1": 0.2568370986920333, "n": 841}, "validation_combined_loss": 2.590989736282425, "seconds": 55.1}

## 5. Export new checkpoint files
The original checkpoint metadata and architecture fields are preserved. Only `state` is replaced, and the output filenames are new.

In [ ]:
def export_state(model):
    return {k:v.detach().cpu().tolist() for k,v in model.state_dict().items()}
info={"method":"search-supervised policy/value fine-tuning","dataset":os.path.basename(dataset_path),
      "positions":len(rows),"trainPositions":len(train_rows),"validationPositions":len(val_rows),
      "epochs":3,"notAlphaZero":True}
cnn_export=copy.deepcopy(cnn_json); cnn_export["state"]=export_state(cnn); cnn_export["trainingInfo"]=info
gnn_export=copy.deepcopy(gnn_json); gnn_export["state"]=export_state(gnn); gnn_export["trainingInfo"]=info
cnn_out="tryaline_resnet_finetuned_v1.json"; gnn_out="tryaline_gnn_finetuned_v1.json"
with open(cnn_out,"w",encoding="utf-8") as f: json.dump(cnn_export,f,separators=(",",":"))
with open(gnn_out,"w",encoding="utf-8") as f: json.dump(gnn_export,f,separators=(",",":"))
print(cnn_out,os.path.getsize(cnn_out),"bytes")
print(gnn_out,os.path.getsize(gnn_out),"bytes")

tryaline_resnet_finetuned_v1.json 893318 bytes
tryaline_gnn_finetuned_v1.json 969125 bytes


In [ ]:
files.download(cnn_out)
files.download(gnn_out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Before using these checkpoints in the app
Training loss and agreement with the search teacher do not prove better playing strength. Next, test tensor compatibility and legal action selection, then run fixed-seed matches against the original models on 3×3, 5×5, and 7×7. Promote only if the evaluation supports it.